# colab_09 — Annotate cluster 0 (decision: vRG vs CP / stress)

## Motivation

`colab_08` integrated balanced 100k + 100k subsamples (Bhaduri 2020 organoids + Bhaduri 2021 fetal cortex) via Harmony. Visual biology was preserved (cortical lineage PAX6 → EOMES → TBR1 traced right → center → bottom-left on UMAP), but the cluster × dataset composition raised a flag: **ca. 41% of each dataset's cells live in >95%-pure clusters**, well above the biology-only ceiling of ca. 10–15%.

The single biggest red flag is **cluster 0**: 23,737 cells, 98.7% Bhaduri 2020 (organoid)-pure, sitting in the cortical RG zone (PAX6⁺ SOX2⁺ MKI67⁺ per 9c). Bhaduri 2021 has 9.8% RG in `cell_type_coarse` (ca. 9,800 fetal RG expected); they should partially populate this cluster but don't.

Three hypotheses for cluster 0:

1. **vRG / cortical RG** — a shared cell type that Harmony under-corrected. Means the mixing diagnostic failed and we need remediation (intersection-only HVGs first, then theta increase, then scVI).
2. **Choroid plexus** — organoid-only off-target tissue (TTR⁺, HTR2C⁺). 2020's protocol-X subset was known to throw choroid plexus contamination per old colab_03. This would be real biology, proceed.
3. **Stress / dying** — organoid-specific stress population (HSPA1A⁺, DDIT3⁺, FOS⁺ AP-1). Also real biology of the 3D culture, proceed with caveats.

## Scope

Focused diagnostic for cluster 0 only. Full 21-cluster annotation deferred to colab_10 once cluster 0's verdict tells us whether colab_08 needs re-running.

## Inputs / outputs

- **Input**: `data/processed/integrated/integrated_100k_harmony.h5ad` (5.54 GB, 200000 × 2000 HVGs in `.X`, full 16,768 genes in `.raw`, leiden in obs).
- **Outputs**: marker tables + plots in `figures/` (Drive). No new h5ad written — this is read-only diagnostic.

## 0. Setup

### 0a — Install dependencies, mount Drive, import packages

Installs `scanpy` and `leidenalg` (not pre-installed on standard Colab images). Mounts Google Drive, imports the scanpy stack, defines `PATHS`.

In [ ]:
!pip install -q scanpy leidenalg

from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=80, frameon=False, figsize=(6, 5))

DRIVE_ROOT = '/content/drive/MyDrive/brain-organoid-trajectories'
PATHS = {
    'integrated': os.path.join(DRIVE_ROOT, 'data/processed/integrated/integrated_100k_harmony.h5ad'),
    'figures':    os.path.join(DRIVE_ROOT, 'figures/colab_09'),
}
os.makedirs(PATHS['figures'], exist_ok=True)
sc.settings.figdir = PATHS['figures']
print('scanpy', sc.__version__, '| anndata', ad.__version__)

## 1. Load integrated object and locate cluster 0

### 1a — Read `integrated_100k_harmony.h5ad`

Loads the integrated AnnData and confirms the expected layout: `.X` = scaled HVGs (200000 × 2000), `.raw` = log-normalized full-gene matrix (200000 × 16,768), `obs['leiden']` and `obs['dataset']` populated.

In [ ]:
adata = sc.read_h5ad(PATHS['integrated'])

print(f'Shape (X):   {adata.shape[0]:,} cells x {adata.shape[1]:,} HVGs')
print(f'Shape (raw): {adata.raw.shape[0]:,} cells x {adata.raw.shape[1]:,} genes')
print(f'obs columns: {list(adata.obs.columns)}')
print(f'obsm keys:   {list(adata.obsm.keys())}')
print()
print('Dataset balance:')
print(adata.obs['dataset'].value_counts())
print()
print(f'Leiden clusters (n = {adata.obs["leiden"].nunique()}): '
      f'{sorted(adata.obs["leiden"].unique(), key=int)}')

### 1b — Cluster 0 size and dataset composition

Verifies the colab_08 finding: cluster 0 size ca. 23,737, ca. 98.7% Bhaduri 2020. Adds context: what fraction of each dataset is in cluster 0, and how does the small Bhaduri 2021 contingent compare to the ca. 9,800 fetal RG cells expected from `cell_type_coarse` in the original Bhaduri 2021 100k subsample.

In [ ]:
cluster_id = '0'
mask0 = adata.obs['leiden'] == cluster_id
n_total = mask0.sum()

dataset_counts = adata.obs.loc[mask0, 'dataset'].value_counts()
n_2020 = int(dataset_counts.get('bhaduri_2020', 0))
n_2021 = int(dataset_counts.get('bhaduri_2021', 0))

# fraction of each dataset that is in cluster 0
frac_of_2020 = n_2020 / (adata.obs['dataset'] == 'bhaduri_2020').sum()
frac_of_2021 = n_2021 / (adata.obs['dataset'] == 'bhaduri_2021').sum()

print(f'Cluster {cluster_id}: {n_total:,} cells total')
print(f'  bhaduri_2020: {n_2020:,} ({n_2020/n_total*100:.1f}%)  '
      f'= {frac_of_2020*100:.1f}% of all 2020 cells')
print(f'  bhaduri_2021: {n_2021:,} ({n_2021/n_total*100:.1f}%)  '
      f'= {frac_of_2021*100:.1f}% of all 2021 cells')
print()
print(f'Reference: Bhaduri 2021 cell_type_coarse RG fraction = 9.8% '
      f'→ ca. 9,800 fetal RG expected to populate the cortical RG cluster')

### 1c — Highlight cluster 0 on the UMAP

Renders the integrated UMAP with only cluster 0 cells colored (others greyed). This visually anchors where cluster 0 sits relative to the lineage paths identified in colab_08 9c.

In [ ]:
sc.pl.umap(
    adata,
    color='leiden',
    groups=[cluster_id],
    size=8,
    legend_loc='right margin',
    title=f'Cluster {cluster_id} highlighted (n = {n_total:,})',
    save=f'_cluster{cluster_id}_highlight.png',
    show=True,
)

## 2. Top differential markers — cluster 0 vs rest

### 2a — Run `rank_genes_groups` on cluster 0 only

Wilcoxon test against the union of all other clusters. `use_raw=True` so we test on the full 16,768-gene log-normalized matrix (some choroid plexus and stress markers may not be in the 2,000 HVGs). Restricting to `groups=['0']` keeps runtime to a few minutes on standard Colab.

In [ ]:
sc.tl.rank_genes_groups(
    adata,
    groupby='leiden',
    groups=[cluster_id],
    reference='rest',
    method='wilcoxon',
    use_raw=True,
    n_genes=200,
)
print(f'rank_genes_groups complete for cluster {cluster_id}')

### 2b — Print top 30 markers (gene, logfoldchange, p-value)

Tabulates gene name, log2 fold change, adjusted p-value. The signature here is the primary disambiguator: vRG markers (PAX6, SOX2, VIM, NES, HES1, FABP7, SLC1A3) vs choroid plexus markers (TTR, HTR2C, CLIC6, AQP1) vs stress markers (HSPA1A, HSPA1B, DDIT3).

In [ ]:
rg = adata.uns['rank_genes_groups']
top = pd.DataFrame({
    'gene':            [r[cluster_id] for r in rg['names'][:30]],
    'log2foldchange':  [r[cluster_id] for r in rg['logfoldchanges'][:30]],
    'pval_adj':        [r[cluster_id] for r in rg['pvals_adj'][:30]],
    'pct_in_cluster':  [r[cluster_id] for r in rg['pts'][:30]] if 'pts' in rg else [None]*30,
})
print(f'Top 30 markers for cluster {cluster_id} vs rest:')
print(top.to_string(index=False))

### 2c — Dotplot of top 15 markers across all 21 clusters

Visualizes how the cluster-0-defining markers express across every cluster. If markers are tightly cluster-0-specific (one column lit up, others dim), cluster 0 is a distinct lineage. If they smear across multiple clusters, cluster 0 is a stress / cell-state signature that overlays a lineage.

In [ ]:
sc.pl.rank_genes_groups_dotplot(
    adata,
    groupby='leiden',
    n_genes=15,
    use_raw=True,
    save=f'_cluster{cluster_id}_top15_dotplot.png',
    show=True,
)

## 3. Targeted lineage panels — vRG, choroid plexus, stress

Three pre-specified marker sets, one per hypothesis. Targeted panels complement the unbiased ranking in section 2 — sometimes the unbiased top-30 is dominated by ribosomal / mitochondrial genes that don't disambiguate the lineage, so we explicitly probe.

- **vRG / cortical RG**: PAX6, SOX2, VIM, NES, HES1, HES5, NOTCH1, FABP7, SLC1A3, MKI67. SLC1A3 (GLAST) and FABP7 (BLBP) are the cleanest cortical RG markers; HES1/HES5 are Notch effectors maintaining stemness; MKI67 confirms cycling state.
- **Choroid plexus**: TTR, HTR2C, CLIC6, AQP1, KRT18, F5, KL, OTX2. TTR (transthyretin) is the gold-standard CP marker — secreted into CSF, near-exclusive to CP epithelium. HTR2C is highly CP-enriched. OTX2 is CP+ but also expressed elsewhere.
- **Stress / dying**: HSPA1A, HSPA1B, DNAJB1, FOS, JUN, ATF3, DDIT3, HSPB1. HSPA1A/B (HSP70) and DNAJB1 (HSP40) flag heat-shock; DDIT3 (CHOP) flags ER stress; FOS / JUN are AP-1 immediate-early.

### 3a — Dotplot: lineage panels × all 21 clusters

Single combined dotplot with `var_group_labels` separating the three panels. Reading: cluster 0 is **vRG** if it's high on PAX6/SOX2/VIM/NES/SLC1A3 but low on TTR/HTR2C and HSPA1A/HSPA1B. **CP** if it's high on TTR/HTR2C/CLIC6/AQP1. **Stress** if it's high on HSPA1A/HSPA1B/DDIT3.

In [ ]:
vrg_markers    = ['PAX6', 'SOX2', 'VIM', 'NES', 'HES1', 'HES5', 'NOTCH1', 'FABP7', 'SLC1A3', 'MKI67']
cp_markers     = ['TTR', 'HTR2C', 'CLIC6', 'AQP1', 'KRT18', 'F5', 'KL', 'OTX2']
stress_markers = ['HSPA1A', 'HSPA1B', 'DNAJB1', 'FOS', 'JUN', 'ATF3', 'DDIT3', 'HSPB1']

# keep only genes present in raw var_names
all_genes = set(adata.raw.var_names)
vrg    = [g for g in vrg_markers    if g in all_genes]
cp     = [g for g in cp_markers     if g in all_genes]
stress = [g for g in stress_markers if g in all_genes]

missing = (set(vrg_markers) - set(vrg)) | (set(cp_markers) - set(cp)) | (set(stress_markers) - set(stress))
if missing:
    print(f'Missing from .raw.var_names (skipped): {sorted(missing)}')

panel = {'vRG / RG': vrg, 'Choroid plexus': cp, 'Stress / IEG': stress}
sc.pl.dotplot(
    adata,
    var_names=panel,
    groupby='leiden',
    use_raw=True,
    standard_scale='var',
    save=f'_cluster{cluster_id}_lineage_panels_dotplot.png',
    show=True,
)

### 3b — UMAP feature plots: vRG markers (combined grid + per-marker individuals)

Combined grid for at-a-glance comparison + individual full-size plots per marker (project rule `feedback_plots_individual.md` — multi-panel grids must be accompanied by per-panel full-size renders to avoid thumbnail misreads).

In [ ]:
# combined grid
sc.pl.umap(
    adata,
    color=vrg,
    use_raw=True,
    ncols=4,
    save=f'_cluster{cluster_id}_vrg_grid.png',
    show=True,
)

# per-marker full-size
for g in vrg:
    sc.pl.umap(
        adata,
        color=g,
        use_raw=True,
        size=8,
        save=f'_cluster{cluster_id}_vrg_{g}.png',
        show=True,
    )

### 3c — UMAP feature plots: choroid plexus markers (grid + individuals)

In [ ]:
sc.pl.umap(
    adata,
    color=cp,
    use_raw=True,
    ncols=4,
    save=f'_cluster{cluster_id}_cp_grid.png',
    show=True,
)

for g in cp:
    sc.pl.umap(
        adata,
        color=g,
        use_raw=True,
        size=8,
        save=f'_cluster{cluster_id}_cp_{g}.png',
        show=True,
    )

### 3d — UMAP feature plots: stress markers (grid + individuals)

In [ ]:
sc.pl.umap(
    adata,
    color=stress,
    use_raw=True,
    ncols=4,
    save=f'_cluster{cluster_id}_stress_grid.png',
    show=True,
)

for g in stress:
    sc.pl.umap(
        adata,
        color=g,
        use_raw=True,
        size=8,
        save=f'_cluster{cluster_id}_stress_{g}.png',
        show=True,
    )

## 4. Quantitative cluster 0 lineage scores

### 4a — Mean panel expression: cluster 0 vs all-others

For each panel (vRG, CP, stress), compute the mean log-normalized expression across panel genes inside cluster 0 and compare to the same mean computed across all other clusters. The panel with the largest positive cluster-0-vs-rest delta is the dominant signature.

In [ ]:
def panel_mean(adata_obj, gene_list, mask):
    """Mean of panel-gene mean expression across cells in mask, on raw (log-normalized)."""
    idx = [adata_obj.raw.var_names.get_loc(g) for g in gene_list]
    sub = adata_obj.raw.X[mask, :][:, idx]  # (n_cells, n_panel_genes)
    if hasattr(sub, 'toarray'):
        sub = sub.toarray()
    return float(sub.mean())

mask_in   = (adata.obs['leiden'] == cluster_id).values
mask_out  = ~mask_in

scores = []
for name, genes in [('vRG / RG', vrg), ('Choroid plexus', cp), ('Stress / IEG', stress)]:
    in_mean  = panel_mean(adata, genes, mask_in)
    out_mean = panel_mean(adata, genes, mask_out)
    scores.append({
        'panel': name,
        'cluster_0_mean': in_mean,
        'rest_mean':      out_mean,
        'delta':          in_mean - out_mean,
        'n_genes':        len(genes),
    })

scores_df = pd.DataFrame(scores)
print('Mean panel expression (log-normalized) — cluster 0 vs rest:')
print(scores_df.to_string(index=False))

### 4b — Per-dataset breakdown inside cluster 0

The 1.3% Bhaduri 2021 contingent inside cluster 0 (ca. 312 cells) is the diagnostic minority. If those 312 cells score high on vRG markers, they really are fetal RG that landed in the same cluster as organoid RG — confirming cluster 0 is a shared cell type and Harmony under-corrected the population imbalance. If they score low on vRG (and high on something else), the 2021 contingent is a different cell type that happens to embed nearby, and cluster 0 is organoid-only biology.

In [ ]:
mask_in_2020 = mask_in & (adata.obs['dataset'] == 'bhaduri_2020').values
mask_in_2021 = mask_in & (adata.obs['dataset'] == 'bhaduri_2021').values

print(f'Cluster 0 — bhaduri_2020 cells: {mask_in_2020.sum():,}')
print(f'Cluster 0 — bhaduri_2021 cells: {mask_in_2021.sum():,}')
print()

rows = []
for name, genes in [('vRG / RG', vrg), ('Choroid plexus', cp), ('Stress / IEG', stress)]:
    rows.append({
        'panel': name,
        '2020_mean':  panel_mean(adata, genes, mask_in_2020),
        '2021_mean':  panel_mean(adata, genes, mask_in_2021),
        'rest_mean':  panel_mean(adata, genes, mask_out),
    })
print('Mean panel expression — split by dataset within cluster 0:')
print(pd.DataFrame(rows).to_string(index=False))

## 5. Decision

### 5a — Classification summary and next-step routing

Pulls section 2 (top-30 markers) and section 4 (panel deltas + per-dataset breakdown) into a single printout that names the dominant signature and points at the next notebook.

Decision rule:
- **vRG dominates** → cluster 0 is a shared cell type Harmony failed to mix. Path: re-run colab_08 with intersection-only HVGs (drop from 2,000 to the 751 HV-in-both subset) as the cheapest test. If still pure, increase Harmony `theta`. If still pure, switch to scVI.
- **CP dominates** → cluster 0 is organoid-only choroid plexus contamination (well-documented in Bhaduri 2020 protocol-X). Real biology, proceed to colab_10 full annotation with caveats about composition imbalance.
- **Stress dominates** → cluster 0 is an organoid stress / dying population. Real biology of 3D culture, proceed to colab_10 with a note that stress cells should likely be filtered before trajectory analysis.
- **Mixed / no clear winner** → run remediation anyway (cheapest test = intersection HVGs), then re-evaluate.

In [ ]:
# pull the top-5 differential markers
top5 = top['gene'].head(5).tolist()

# panel deltas from section 4a
deltas = {row['panel']: row['delta'] for row in scores}
ranked = sorted(deltas.items(), key=lambda kv: kv[1], reverse=True)

print('=' * 70)
print(f'CLUSTER {cluster_id} CLASSIFICATION SUMMARY')
print('=' * 70)
print(f'Size:               {n_total:,} cells')
print(f'Dataset balance:    {n_2020:,} 2020 ({n_2020/n_total*100:.1f}%) | '
      f'{n_2021:,} 2021 ({n_2021/n_total*100:.1f}%)')
print(f'Top 5 markers:      {", ".join(top5)}')
print()
print('Panel deltas (cluster 0 mean − rest mean):')
for panel_name, delta in ranked:
    print(f'  {panel_name:20s} {delta:+.4f}')
print()
print(f'Dominant signature: {ranked[0][0]}')
print()
print('Next step:')
winner = ranked[0][0]
if winner.startswith('vRG'):
    print('  → Harmony under-corrected. Re-run colab_08 with intersection-only HVGs')
    print('    (drop from 2,000 to the 751 HV-in-both subset). Cheapest test of the')
    print('    same hypothesis before reaching for theta or scVI.')
elif winner.startswith('Choroid'):
    print('  → Cluster 0 is organoid-only CP contamination. Real biology.')
    print('    Proceed to colab_10 (full 21-cluster annotation) with composition caveats.')
elif winner.startswith('Stress'):
    print('  → Cluster 0 is an organoid stress / dying population. Real biology.')
    print('    Proceed to colab_10. Likely filter before trajectory analysis.')
print('=' * 70)